In [54]:
import torch
import cv2
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from retinaface import RetinaFace
from tqdm import tqdm
import os

In [55]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [56]:
model, transform = torch.hub.load('fkryan/gazelle', 'gazelle_dinov2_vitl14_inout')

Using cache found in C:\Users\phile/.cache\torch\hub\fkryan_gazelle_main
Using cache found in C:\Users\phile/.cache\torch\hub\facebookresearch_dinov2_main


In [57]:
model.eval()
model.to(device)

GazeLLE(
  (backbone): DinoV2Backbone(
    (model): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (blocks): ModuleList(
        (0-23): 24 x NestedTensorBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): MemEffAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (act): GELU(approximate='none')
            (fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (drop): Dropout(p=0.0, inpla

In [58]:
colors = ['yellow', 'red', 'green', 'blue', 'lime']

In [59]:
def process_frame(frame):
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(frame_rgb)
    width, height = img.size

    faces = RetinaFace.detect_faces(frame_rgb)
    if not isinstance(faces, dict):
        return frame

    bboxes = [faces[key]['facial_area'] for key in faces.keys()]
    norm_bboxes = [[np.array(bbox) / np.array([width, height, width, height]) for bbox in bboxes]]

    img_t = transform(img)
    img_t = img_t.unsqueeze(0).to(device)

    input_data = {
        "images": img_t,
        "bboxes": norm_bboxes
    }

    with torch.no_grad():
        output = model(input_data)

    res_img = viz_all(
            img,
            output['heatmap'][0],
            norm_bboxes[0],
            output['inout'][0] if output['inout'] is not None else None,
            0.5
    )

    res_arr = np.array(res_img)
    return cv2.cvtColor(res_arr, cv2.COLOR_RGB2BGR)


In [60]:
def viz_all(pil_img, heatmaps, bboxes, inout_scores, inout_th):
    over_img = pil_img.convert('RGBA')
    draw = ImageDraw.Draw(over_img)
    width, height = pil_img.size

    for i in range(len(bboxes)):
        bbox = bboxes[i]
        xmin, ymin, xmax, ymax = bbox
        color = colors[i % len(colors)]

        draw.rectangle([xmin * width, ymin * height, xmax * width, ymax * height], outline=color, width = int(min(width, height) / 100))

        if inout_scores is not None:
            inout_score = inout_scores[i]

            text = f"frame: {inout_score:.2f}"
            text_y = ymax * height + int(height / 100)
            draw.text((xmin * width, text_y), text, fill=color)

            if inout_score > inout_th:
                heatmap = heatmaps[i]
                heatmap_arr = heatmap.cpu().numpy()
                max_idx = np.unravel_index(np.argmax(heatmap_arr), heatmap_arr.shape)

                gz_target_x = max_idx[1] / heatmap_arr.shape[1] * width
                gz_target_y = max_idx[0] / heatmap_arr.shape[0] * height
                bbox_center_x = (xmin + xmax) / 2 * width
                bbox_center_y = (ymin + ymax) / 2 * height

                draw.ellipse([(gz_target_x - 5, gz_target_y - 5), (gz_target_x + 5, gz_target_y + 5)], fill=color, width=int(0.005 * min(width, height)))
                draw.line([(bbox_center_x, bbox_center_y), (gz_target_x, gz_target_y)], fill=color, width = int(0.005 * min(width, height)))

    return over_img.convert('RGB')


In [61]:
def process_video(input_video, output_path, start_time=0, duration=None):
    cap = cv2.VideoCapture(input_video)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    start_time = int(start_time * fps)
    if duration:
        end_frame = start_time + int(duration * fps)
    else:
        end_frame = total_frames

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_time)

    try:
        with tqdm(total=end_frame - start_time, desc="Processing video") as pbar:
            frame_count = start_time
            while cap.isOpened() and frame_count < end_frame:
                ret, frame = cap.read()
                if not ret:
                    break

                processed_frame = process_frame(frame)
                out.write(processed_frame)

                frame_count += 1
                pbar.update(1)
    finally:
        cap.release()
        out.release()
        cv2.destroyAllWindows()
        

In [62]:
input_video = "teach.mp4"
output_video = "output_detection_teach.mp4"

In [63]:
process_video(input_video, output_video)

Processing video: 100%|██████████| 486/486 [18:08<00:00,  2.24s/it]
